In [2]:
# ── Cell 1: Mount Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Because of gene selection form 1600 to 1593, it does not work

In [ ]:
# ============================================================
# scGAN / cscGAN — Generate Synthetic Cells
# Google Colab | TensorFlow 2
# ============================================================



# ── Cell 2: Imports ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow: {tf.__version__}')

# ── Cell 3: Configuration — EDIT PATHS ───────────────────────
DATASET = 'PDO'   # 'PBMC' or 'PDO'

WEIGHTS = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scGAN Approach/PBMC_B_MONO_experiments/experiments/pbmc_cscgan_v2/weights/gen_best.weights.h5',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scGAN Approach/PDO_Stem_Diff_experiments/experiments/PDO_cscgan_v1/weights/gen_best.weights.h5',
}

REAL_DATA = {
    'PBMC': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/b_Class_dataset.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/mono_Class_dataset.csv',
    },
    'PDO': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Stem_High_Raw_Finalized.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Differential_Low_Raw_Finalized.csv',
    },
}

OUTPUT_DIR = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scGAN/PBMC',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scGAN/PDO',
}

N_PER_CLASS = {'PBMC': 1186, 'PDO': 1415}

CLASS_NAMES = {
    'PBMC': {0: 'b',         1: 'mono'},
    'PDO' : {0: 'stem_high', 1: 'diff_low'},
}

# Must match training exactly
LATENT_DIM = 128
GEN_LAYERS = [256, 512, 1024]
N_CLASSES  = 2
LSN_TARGET = 20000
SEED       = 42

# ── Cell 4: Architecture ─────────────────────────────────────

class LSNLayer(layers.Layer):
    def __init__(self, target_sum=20000, **kw):
        super().__init__(**kw)
        self.target_sum = float(target_sum)
    def call(self, x):
        row_sum = tf.reduce_sum(x, axis=1, keepdims=True)
        row_sum = tf.maximum(row_sum, 1e-38)
        return (x / row_sum) * self.target_sum
    def get_config(self):
        c = super().get_config()
        c['target_sum'] = self.target_sum
        return c

class ConditionalBatchNorm(layers.Layer):
    def __init__(self, n_features, n_classes, **kw):
        super().__init__(**kw)
        self.n_features = n_features
        self.n_classes  = n_classes
        self.bn         = layers.BatchNormalization(center=False, scale=False)
        self.gamma_emb  = layers.Embedding(n_classes, n_features,
                              embeddings_initializer='ones')
        self.beta_emb   = layers.Embedding(n_classes, n_features,
                              embeddings_initializer='zeros')
    def call(self, x, labels, training=False):
        x     = self.bn(x, training=training)
        gamma = self.gamma_emb(labels)
        beta  = self.beta_emb(labels)
        return x * gamma + beta
    def get_config(self):
        c = super().get_config()
        c.update({'n_features': self.n_features, 'n_classes': self.n_classes})
        return c

def build_generator(n_genes, latent_dim, layer_sizes, n_classes, lsn_target):
    z   = keras.Input(shape=(latent_dim,), name='z')
    lbl = keras.Input(shape=(), dtype='int32', name='label')
    x = z
    for units in layer_sizes:
        x = layers.Dense(units, use_bias=False)(x)
        x = ConditionalBatchNorm(units, n_classes)(x, lbl)
        x = layers.ReLU()(x)
    x = layers.Dense(n_genes, activation='relu')(x)
    x = LSNLayer(target_sum=lsn_target)(x)
    return keras.Model(inputs=[z, lbl], outputs=x, name='cscGAN_generator')

# ── Cell 5: Build & Load Weights ─────────────────────────────

sample  = pd.read_csv(list(REAL_DATA[DATASET].values())[0], header=None, nrows=1)
N_GENES = sample.shape[1]
print(f'Dataset: {DATASET} | n_genes: {N_GENES}')

generator = build_generator(N_GENES, LATENT_DIM, GEN_LAYERS, N_CLASSES, LSN_TARGET)

# Dummy pass to initialise weights before loading
tf.random.set_seed(SEED)
_ = generator([tf.zeros([2, LATENT_DIM]), tf.zeros([2], dtype=tf.int32)], training=False)
print(f'Generator: {generator.count_params():,} parameters')

generator.load_weights(WEIGHTS[DATASET])
print(f'Weights loaded.')

# ── Cell 6: Generate Per Class & Save ────────────────────────

os.makedirs(OUTPUT_DIR[DATASET], exist_ok=True)
tf.random.set_seed(SEED)

for class_idx, class_name in CLASS_NAMES[DATASET].items():
    n = N_PER_CLASS[DATASET]
    z      = tf.random.normal([n, LATENT_DIM], seed=SEED + class_idx)
    labels = tf.fill([n], class_idx)

    synthetic = generator([z, labels], training=False).numpy()

    print(f'\nClass {class_idx} ({class_name})')
    print(f'  Shape   : {synthetic.shape}')
    print(f'  Range   : [{synthetic.min():.2f}, {synthetic.max():.2f}]')
    print(f'  Row sum : {synthetic.sum(axis=1).mean():.0f}  (~{LSN_TARGET})')
    print(f'  Sparsity: {(synthetic == 0).mean()*100:.1f}%')

    out_path = os.path.join(OUTPUT_DIR[DATASET],
                            f'scgan_synthetic_{class_name}_expression.csv')
    pd.DataFrame(synthetic).to_csv(out_path, header=False, index=False)
    print(f'  Saved → {out_path}')

TensorFlow: 2.20.0
Dataset: PDO | n_genes: 1600
Generator: 2,338,880 parameters


ValueError: A total of 1 objects could not be loaded. Example error message for object <Dense name=dense_7, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(1024, 1600), Received: value.shape=(1024, 1593). Target variable: <Variable path=dense_7/kernel, shape=(1024, 1600), dtype=float32, value=[[-0.04549139  0.01296489  0.01732612 ...  0.04155096  0.01686773
  -0.03872637]
 [ 0.0461611  -0.0089697  -0.04556551 ... -0.02646567 -0.04029116
  -0.03194003]
 [ 0.01783312  0.03528329  0.01748231 ... -0.03642242 -0.02126217
   0.01363985]
 ...
 [-0.02471074 -0.0002882  -0.04370602 ... -0.001154   -0.0048804
   0.02831405]
 [ 0.02413852 -0.00107386 -0.0403292  ...  0.01772279 -0.00178508
  -0.00027739]
 [ 0.03077396  0.02389709 -0.0437918  ... -0.00197782  0.02652395
   0.04649371]]>

List of objects that could not be loaded:
[<Dense name=dense_7, built=True>]

Works!

In [5]:
# ============================================================
# scGAN / cscGAN — Generate Synthetic Cells
# Google Colab | TensorFlow 2
# ============================================================

# ── Cell 2: Imports ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow: {tf.__version__}')

# ── Cell 3: Configuration — EDIT PATHS ───────────────────────
DATASET = 'PBMC'   # 'PBMC' or 'PDO'

WEIGHTS = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scGAN Approach/PBMC_B_MONO_experiments/experiments/pbmc_cscgan_v2/weights/gen_best.weights.h5',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scGAN Approach/PDO_Stem_Diff_experiments/experiments/PDO_cscgan_v1/weights/gen_best.weights.h5',
}

REAL_DATA = {
    'PBMC': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/b_Class_dataset.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/mono_Class_dataset.csv',
    },
    'PDO': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Stem_High_Raw_Finalized.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Differential_Low_Raw_Finalized.csv',
    },
}

OUTPUT_DIR = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scGAN/PBMC',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset/scGAN/PDO',
}

N_GENES_MAP = {'PBMC': 1600, 'PDO': 1593}
N_PER_CLASS = {'PBMC': 1186, 'PDO': 1415}

CLASS_NAMES = {
    'PBMC': {0: 'b',         1: 'mono'},
    'PDO' : {0: 'stem_high', 1: 'diff_low'},
}

# Must match training exactly
LATENT_DIM = 128
GEN_LAYERS = [256, 512, 1024]
N_CLASSES  = 2
LSN_TARGET = 20000
N_GENES    = 1593   # PDO was filtered to 1593 genes during training
SEED       = 42

# ── Cell 4: Architecture ─────────────────────────────────────

class LSNLayer(layers.Layer):
    def __init__(self, target_sum=20000, **kw):
        super().__init__(**kw)
        self.target_sum = float(target_sum)
    def call(self, x):
        row_sum = tf.reduce_sum(x, axis=1, keepdims=True)
        row_sum = tf.maximum(row_sum, 1e-38)
        return (x / row_sum) * self.target_sum
    def get_config(self):
        c = super().get_config()
        c['target_sum'] = self.target_sum
        return c

class ConditionalBatchNorm(layers.Layer):
    def __init__(self, n_features, n_classes, **kw):
        super().__init__(**kw)
        self.n_features = n_features
        self.n_classes  = n_classes
        self.bn         = layers.BatchNormalization(center=False, scale=False)
        self.gamma_emb  = layers.Embedding(n_classes, n_features,
                              embeddings_initializer='ones')
        self.beta_emb   = layers.Embedding(n_classes, n_features,
                              embeddings_initializer='zeros')
    def call(self, x, labels, training=False):
        x     = self.bn(x, training=training)
        gamma = self.gamma_emb(labels)
        beta  = self.beta_emb(labels)
        return x * gamma + beta
    def get_config(self):
        c = super().get_config()
        c.update({'n_features': self.n_features, 'n_classes': self.n_classes})
        return c

def build_generator(n_genes, latent_dim, layer_sizes, n_classes, lsn_target):
    z   = keras.Input(shape=(latent_dim,), name='z')
    lbl = keras.Input(shape=(), dtype='int32', name='label')
    x = z
    for units in layer_sizes:
        x = layers.Dense(units, use_bias=False)(x)
        x = ConditionalBatchNorm(units, n_classes)(x, lbl)
        x = layers.ReLU()(x)
    x = layers.Dense(n_genes, activation='relu')(x)
    x = LSNLayer(target_sum=lsn_target)(x)
    return keras.Model(inputs=[z, lbl], outputs=x, name='cscGAN_generator')

# ── Cell 5: Build & Load Weights ─────────────────────────────

N_GENES = N_GENES_MAP[DATASET]
print(f'Dataset: {DATASET} | n_genes: {N_GENES}')

generator = build_generator(N_GENES, LATENT_DIM, GEN_LAYERS, N_CLASSES, LSN_TARGET)

tf.random.set_seed(SEED)
_ = generator([tf.zeros([2, LATENT_DIM]), tf.zeros([2], dtype=tf.int32)], training=False)

generator.load_weights(WEIGHTS[DATASET])   # <-- fix: index with DATASET
print('Weights loaded.')

# ── Cell 6: Generate Per Class & Save ────────────────────────
os.makedirs(OUTPUT_DIR[DATASET], exist_ok=True)  # <-- index with DATASET

for class_idx, class_name in CLASS_NAMES[DATASET].items():  # <-- index with DATASET
    n      = N_PER_CLASS[DATASET]
    z      = tf.random.normal([n, LATENT_DIM], seed=SEED + class_idx)
    labels = tf.fill([n], class_idx)

    synthetic = generator([z, labels], training=False).numpy()

    print(f'\nClass {class_idx} ({class_name})')
    print(f'  Shape   : {synthetic.shape}')
    print(f'  Range   : [{synthetic.min():.2f}, {synthetic.max():.2f}]')
    print(f'  Row sum : {synthetic.sum(axis=1).mean():.0f}  (~{LSN_TARGET})')
    print(f'  Sparsity: {(synthetic == 0).mean()*100:.1f}%')

    out_path = os.path.join(OUTPUT_DIR[DATASET],
                            f'scgan_synthetic_{class_name}_expression.csv')
    pd.DataFrame(synthetic).to_csv(out_path, header=False, index=False)
    print(f'  Saved → {out_path}')

print('\nDone.')

TensorFlow: 2.20.0
Dataset: PBMC | n_genes: 1600
Weights loaded.

Class 0 (b)
  Shape   : (1186, 1600)
  Range   : [0.00, 2008.74]
  Row sum : 20000  (~20000)
  Sparsity: 84.3%
  Saved → /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scGAN/PBMC/scgan_synthetic_b_expression.csv

Class 1 (mono)
  Shape   : (1186, 1600)
  Range   : [0.00, 1723.71]
  Row sum : 20000  (~20000)
  Sparsity: 73.6%
  Saved → /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scGAN/PBMC/scgan_synthetic_mono_expression.csv

Done.
